# 02 — End-to-end synthetic pipeline

Demonstrates the full pipeline on three synthetic cubes:

1. **Signal** — GRMHD + injected TFPT signal: all three nulls should fire, **DETECTION**.
2. **Null** — GRMHD only: no detection.
3. **Systematic** — GRMHD + constant calibration offset: frequency null fires (offset is achromatic), but the **profile** and **sign-flip** nulls reject. No detection.

The three-null protocol filters TFPT signals from miscalibrations. This notebook is the visual proof.

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import numpy as np

from tfpt_eht import (
    BlackHoleGeometry,
    SyntheticConfig,
    compute_residual_intercept,
    deproject_radial,
    generate_observed_mock,
    rotation_measure_fit,
    run_all_nulls,
)
from tfpt_eht.plotting import plot_null_summary, plot_radial_profile, plot_residual_image

In [ ]:
def run_one(cfg, *, tfpt_signal, systematic=0.0):
    cfg_flip = dataclasses.replace(
        cfg,
        geometry=BlackHoleGeometry(
            center_x=cfg.geometry.center_x,
            center_y=cfg.geometry.center_y,
            r_inner=cfg.geometry.r_inner,
            r_outer=cfg.geometry.r_outer,
            sign_orientation=-cfg.geometry.sign_orientation,
        ),
    )
    obs, grmhd = generate_observed_mock(cfg, tfpt_signal=tfpt_signal, systematic_offset=systematic)
    obs_flip, _ = generate_observed_mock(cfg_flip, tfpt_signal=tfpt_signal, systematic_offset=systematic)
    chi0_obs, _, _ = rotation_measure_fit(obs)
    chi0_obs_flip, _, _ = rotation_measure_fit(obs_flip)
    residual = compute_residual_intercept(chi0_obs, grmhd.chi_0)
    residual_flip = compute_residual_intercept(chi0_obs_flip, grmhd.chi_0)
    p_plus = deproject_radial(residual, obs.x, obs.y,
                              r_inner=cfg.geometry.r_inner,
                              r_outer=cfg.geometry.r_outer, n_bins=15)
    p_minus = deproject_radial(residual_flip, obs.x, obs.y,
                               r_inner=cfg.geometry.r_inner,
                               r_outer=cfg.geometry.r_outer, n_bins=15)
    residual_cube = obs.chi - grmhd.cube
    chi_per_chan = np.nanmean(residual_cube, axis=(1, 2))
    sig_per_chan = np.full_like(chi_per_chan, cfg.angle_noise_sigma / cfg.image_size)
    report = run_all_nulls(
        chi_residual_per_channel=chi_per_chan,
        lambda_sq=obs.lambda_sq,
        sigma_per_channel=sig_per_chan,
        profile_plus=p_plus,
        profile_minus=p_minus,
        q_e_eff=cfg.q_e_eff,
        q_m_eff=cfg.q_m_eff,
    )
    return obs, grmhd, residual, p_plus, p_minus, report

## Case 1: signal

In [ ]:
cfg = SyntheticConfig(image_size=128, q_e_eff=8.0, q_m_eff=8.0, angle_noise_sigma=2e-7)
obs, grmhd, residual, p_plus, p_minus, report = run_one(cfg, tfpt_signal=True)
print(f'detection = {report.detection}')
print(f'frequency: passed={report.frequency.passed},  slope sig = {report.frequency.detail["slope_significance_sigma"]:.2f}σ')
print(f'profile:   passed={report.profile.passed},    slope = {report.profile.statistic:.3f},  amp ratio = {report.profile.detail["amplitude_ratio"]:.3f}')
print(f'sign-flip: passed={report.sign_flip.passed},  pearson r = {report.sign_flip.statistic:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
plot_residual_image(residual, obs.x, obs.y, ax=axes[0],
                    title='Signal case: residual intercept')
plot_radial_profile(p_plus, ax=axes[1], q_e_eff=cfg.q_e_eff, q_m_eff=cfg.q_m_eff)
plt.tight_layout()
plt.show()

## Case 2: null

In [ ]:
obs, grmhd, residual, p_plus, p_minus, report = run_one(cfg, tfpt_signal=False)
print(f'detection = {report.detection}')
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.5))
plot_residual_image(residual, obs.x, obs.y, ax=axes[0], title='Null case: residual intercept')
plot_radial_profile(p_plus, ax=axes[1], q_e_eff=cfg.q_e_eff, q_m_eff=cfg.q_m_eff)
plt.tight_layout()
plt.show()

## Case 3: systematic

A constant calibration offset of $\sim 2\,\mu$rad. The frequency null cannot tell this apart from a real signal (offset is achromatic), but the profile and sign-flip nulls catch it.

In [ ]:
obs, grmhd, residual, p_plus, p_minus, report = run_one(cfg, tfpt_signal=False, systematic=2e-6)
print(f'detection = {report.detection}')
print(f'frequency: passed={report.frequency.passed}')
print(f'profile:   passed={report.profile.passed},  slope = {report.profile.statistic:.3f}')
print(f'sign-flip: passed={report.sign_flip.passed},  r = {report.sign_flip.statistic:.3f}')

fig, ax = plt.subplots(figsize=(5.5, 3.5))
plot_null_summary(report, ax=ax)
plt.tight_layout()
plt.show()